# AdaBoost with decision stumps

**CSCI 5612 - Module 4, Project Part 4**

Same label, same 29 features and same train/test split as the Decision Trees notebook. Rounds, learning rate and the decision cut-off are tuned on a validation set carved from the training set; the test set is used once.

In [1]:
import warnings; warnings.filterwarnings('ignore', message='X does not have valid feature names')
import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

pd.set_option('display.width', 200)
RANDOM_STATE, TEST_SIZE = 5612, 0.25

## 1. Labelled data, then one-hot encoding

Segment and component are text, so each category becomes its own 0/1 column.

In [2]:
c = pd.read_csv('../data/clean/complaints_clean.csv.gz', low_memory=False)
my = pd.read_csv('../data/clean/vehicle_model_year.csv')
c = c.merge(my[['make','model','model_year','OverallRating','RolloverPossibility','has_fcw','has_ldw','n_recalls']],
            on=['make','model','model_year'], how='inner')
c['label'] = np.where(c['crash'], 'Crash', 'No crash')
top = c['component_primary'].value_counts().head(12).index
c['component'] = np.where(c['component_primary'].isin(top), c['component_primary'], 'OTHER')
base = pd.DataFrame({'vehicle_age': c.vehicle_age_at_incident, 'days_to_report': c.days_to_report,
    'model_year': c.model_year, 'overall_rating': c.OverallRating, 'rollover_prob': c.RolloverPossibility,
    'has_fcw': c.has_fcw, 'has_ldw': c.has_ldw, 'n_recalls': c.n_recalls,
    'summary_words': c.summary.fillna('').astype(str).str.split().str.len()})
print('before encoding:'); display(c[['segment','component','label']].head(3))
feat = pd.concat([base, pd.get_dummies(c.segment, prefix='seg').astype(int),
                  pd.get_dummies(c.component, prefix='cmp').astype(int)], axis=1)
feat['label'] = c['label'].values
feat = feat.dropna().reset_index(drop=True)
print(feat.shape); feat.iloc[:3, 9:20]

before encoding:


,segment,component,label
0,Sedan,VISIBILITY/WIPER,No crash
1,Sedan,ELECTRICAL SYSTEM,No crash
2,Sedan,UNKNOWN OR OTHER,No crash


(80615, 30)


,seg_Electric,seg_Hybrid,seg_Luxury,seg_Minivan,seg_SUV,seg_Sedan,seg_Truck,cmp_AIR BAGS,cmp_ELECTRICAL SYSTEM,cmp_ENGINE,cmp_FORWARD COLLISION AVOIDANCE
0,0,0,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,1,0,0,1,0,0
2,0,0,0,0,0,1,0,0,0,0,0


## 2. Train / validation / test

In [3]:
X, y = feat.drop(columns='label'), feat['label']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
Xfit, Xval, yfit, yval = train_test_split(Xtr, ytr, test_size=0.2, random_state=RANDOM_STATE, stratify=ytr)
assert not set(Xtr.index) & set(Xte.index) and not set(Xfit.index) & set(Xval.index)
print(f'fit {len(Xfit):,}  validation {len(Xval):,}  test {len(Xte):,}')

fit 48,368  validation 12,093  test 20,154


## 3. Validation performance as rounds are added

One fit per learning rate; `staged_decision_function` gives the score after every round.

In [4]:
def stumps(n, lr):
    return AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=n,
                              learning_rate=lr, random_state=RANDOM_STATE)

yv = (yval == 'Crash').astype(int).values
w = compute_sample_weight('balanced', yfit)
rows = []
for lr in [0.25, 0.5, 1.0]:
    m = stumps(400, lr).fit(Xfit, yfit, sample_weight=w)
    sgn = 1 if list(m.classes_).index('Crash') == 1 else -1
    for k, s in enumerate(m.staged_decision_function(Xval), start=1):
        rows.append({'lr': lr, 'rounds': k, 'val_acc': ((sgn*s > 0) == yv).mean(),
                     'val_ap': average_precision_score(yv, sgn*s)})
curves = pd.DataFrame(rows)
curves[curves.rounds.isin([1, 10, 50, 100, 126, 200, 400])].pivot(index='rounds', columns='lr', values=['val_acc','val_ap']).round(4)

val_acc                  val_ap                
lr        0.25    0.50    1.00    0.25    0.50    1.00
rounds                                                
1       0.9434  0.9434  0.9434  0.1516  0.1516  0.1516
10      0.9434  0.9434  0.7438  0.1516  0.1975  0.2339
50      0.9224  0.7921  0.7755  0.2733  0.2917  0.3179
100     0.7882  0.7967  0.7771  0.2958  0.3043  0.3222
126     0.7850  0.7839  0.7640  0.2952  0.3144  0.3304
200     0.7874  0.7806  0.7814  0.3103  0.3238  0.3282
400     0.7944  0.7881  0.7801  0.3235  0.3267  0.3297

In [5]:
top = curves.val_ap.max()
best = curves[curves.val_ap >= top - 0.001].sort_values(['rounds','val_ap'], ascending=[True, False]).iloc[0]
best

lr           1.000000
rounds     126.000000
val_acc      0.763996
val_ap       0.330386
Name: 925, dtype: float64

## 4. Choosing the cut-off on validation, then one test evaluation

In [6]:
lr, n = float(best.lr), int(best.rounds)
m = stumps(n, lr).fit(Xfit, yfit, sample_weight=w)
sgn = 1 if list(m.classes_).index('Crash') == 1 else -1
p, r, t = precision_recall_curve(yv, sgn * m.decision_function(Xval))
f = 2*p[:-1]*r[:-1] / np.maximum(p[:-1]+r[:-1], 1e-12)
thr = t[f.argmax()]
print(f'cut-off {thr:.4f}, validation F1 {f.max():.4f}')

cut-off 0.3857, validation F1 0.3620


In [7]:
ada = stumps(n, lr).fit(Xtr, ytr, sample_weight=compute_sample_weight('balanced', ytr))
sgn = 1 if list(ada.classes_).index('Crash') == 1 else -1
pa = np.where(sgn * ada.decision_function(Xte) > thr, 'Crash', 'No crash')
tree = DecisionTreeClassifier(criterion='gini', max_depth=3, class_weight='balanced',
                              random_state=RANDOM_STATE).fit(Xtr, ytr)
rows = []
for name, pr in [(f'AdaBoost {n} stumps', pa), ('Part 3 tree', tree.predict(Xte))]:
    rows.append({'model': name, 'accuracy': accuracy_score(yte, pr),
                 'precision': precision_score(yte, pr, pos_label='Crash'),
                 'recall': recall_score(yte, pr, pos_label='Crash'), 'f1': f1_score(yte, pr, pos_label='Crash')})
pd.DataFrame(rows).round(4)

,model,accuracy,precision,recall,f1
0,AdaBoost 126 stumps,0.9481,0.4281,0.2944,0.3489
1,Part 3 tree,0.8822,0.1992,0.4953,0.2841


In [8]:
pd.DataFrame(confusion_matrix(yte, pa, labels=['No crash','Crash']),
             index=['actual No crash','actual Crash'], columns=['pred No crash','pred Crash'])

,pred No crash,pred Crash
actual No crash,18829,374
actual Crash,671,280


## 5. The first stumps and their weights

In [9]:
pd.DataFrame({'round': range(1, 9),
              'feature': [X.columns[s.tree_.feature[0]] for s in ada.estimators_[:8]],
              'threshold': [s.tree_.threshold[0] for s in ada.estimators_[:8]],
              'weighted_error': ada.estimator_errors_[:8], 'alpha': ada.estimator_weights_[:8]}).round(4)

,round,feature,threshold,weighted_error,alpha
0,1,cmp_AIR BAGS,0.5,0.3596,0.5770
1,2,cmp_AIR BAGS,0.5,0.3977,0.4149
2,3,cmp_AIR BAGS,0.5,0.4222,0.3136
3,4,vehicle_age,3.5,0.3858,0.4650
4,5,cmp_ENGINE,0.5,0.4263,0.2970
5,6,cmp_VEHICLE SPEED CONTROL,0.5,0.4309,0.2780
6,7,cmp_AIR BAGS,0.5,0.4689,0.1244
7,8,cmp_AIR BAGS,0.5,0.4494,0.2031
